# ATC Multi-Agent GRPO Training — Jupyter Server

**Run order:** top to bottom. Edit **§0 Config** first.

**Default pipeline:** **§8 smoke** — one SFT step + one GRPO step (cheap wiring). **§9 ablation** — full SFT (~400 steps) saved to `SFT_OUTPUT_DIR`, then **two** GRPO runs with the same `GRPO_ABLATION_MAX_STEPS`: **from base** (no SFT adapter) → `GRPO_FROM_BASE_DIR`, and **from SFT** → `GRPO_FROM_SFT_DIR`, so you can compare evals and plots against the same baseline. With **two GPUs**, both GRPO jobs can run in parallel (`ABLATION_PARALLEL_GRPO`). **§10** plots each run plus an ablation overlay.

**Abridged path (opt-in only):** set `STATIC_GROUNDED_DATASET = True` for a pre-built static HF list (`--static_grounded_dataset`), or `RELAX_ROSTER = True` to disable strict roster asserts — for tiny smoke / debugging only, not the default.

## 0. Config — edit before running

In [ ]:
from pathlib import Path
import os


def _find_repo_root() -> Path:
    """CWD may be repo root or `training/`; locate directory containing `training/train_grpo.py`."""
    p = Path(".").resolve()
    for _ in range(6):
        if (p / "training" / "train_grpo.py").is_file():
            return p
        if p.parent == p:
            break
        p = p.parent
    return Path(".").resolve()


# ── Paths ──────────────────────────────────────────────────────────────────
REPO_DIR = _find_repo_root()
TRAIN_GRPO_SCRIPT = REPO_DIR / "training" / "train_grpo.py"
TRAIN_SFT_SCRIPT  = REPO_DIR / "training" / "train_sft.py"
OUTPUT_DIR = Path("/tmp/atc/outputs")     # change to a persistent path if needed
LOGS_DIR   = Path("/tmp/atc/logs")

# ── Model & training ───────────────────────────────────────────────────────
SMOKE_MODEL   = "Qwen/Qwen2.5-7B-Instruct"   # §8 smoke only (same stack, fewer steps)
TRAIN_MODEL   = "Qwen/Qwen2.5-7B-Instruct"
EPISODES      = 80
N_GENERATIONS = 4
SEED          = 42

# §8 = 1×SFT + 1×GRPO. §9 = full SFT + two GRPO ablations (base vs SFT init).
RUN_SMOKE_PIPELINE = True     # False = skip §8 entirely
RUN_ABLATION_MAIN  = True     # False = skip §9 (only smoke, or nothing if smoke off)
QUICK_DEV = False              # True → only tiny smoke + tiny §9 (SFT=1, GRPO ablation=1); turns RUN_ABLATION_MAIN off

# A100-class VRAM: larger micro-batch (multiple of 6 for roster). T4: set HIGH_VRAM_MODE = False.
HIGH_VRAM_MODE = True
try:
    import torch as _torch_vram
    _N_GPU = int(_torch_vram.cuda.device_count()) if _torch_vram.cuda.is_available() else 0
except Exception:
    _N_GPU = 0
ABLATION_PARALLEL_GRPO = _N_GPU >= 2  # two GRPO subprocesses on CUDA 0 and 1

BATCH_SIZE      = 12 if HIGH_VRAM_MODE else 8
GRAD_ACCUM      = 2
SFT_BATCH       = 4 if HIGH_VRAM_MODE else 2
SFT_GRAD_ACCUM  = 4 if not HIGH_VRAM_MODE else 2

# Grounded + continuous curriculum + full multi-agent roster
USE_GROUNDED_CURRICULUM = True
STATIC_GROUNDED_DATASET = False
RELAX_ROSTER = False
CURRICULUM_STATE = None

RUN_SFT_PHASE     = True
SFT_OUTPUT_DIR    = OUTPUT_DIR / "atc-sft-json"
SFT_N_EPISODES    = 120
SFT_MAX_STEPS     = 400

# Same cap for both GRPO ablation runs (``min(live_max_steps, GRPO_ABLATION_MAX_STEPS)``). None = no cap.
GRPO_ABLATION_MAX_STEPS = 200

MAX_NEW_TOKENS  = 384
TEMPERATURE     = 0.7
LOGGING_STEPS   = 1
EVAL_EPISODES   = 3
STRICT_GATES    = True

WANDB_KEY    = ""
WANDB_PROJECT = "atc-multiagent-grpo"

if QUICK_DEV:
    RUN_ABLATION_MAIN = False
    RUN_SMOKE_PIPELINE = True
    SFT_MAX_STEPS = 1
    GRPO_ABLATION_MAX_STEPS = 1

# ── Derived paths ───────────────────────────────────────────────────────────
SMOKE_OUTPUT_DIR   = OUTPUT_DIR / "atc-smoke"
SMOKE_SFT_DIR      = SMOKE_OUTPUT_DIR / "sft-smoke"
SMOKE_GRPO_DIR     = SMOKE_OUTPUT_DIR / "grpo-smoke"
GRPO_FROM_BASE_DIR = OUTPUT_DIR / "grpo-from-base"
GRPO_FROM_SFT_DIR  = OUTPUT_DIR / "grpo-from-sft"
TRAIN_OUTPUT_DIR  = GRPO_FROM_SFT_DIR   # default artifact focus (SFT-init GRPO)
PLOTS_DIR         = OUTPUT_DIR / "plots"
PLOTS_ABLATION_DIR = PLOTS_DIR / "ablation"

_mkdirs = (
    OUTPUT_DIR, LOGS_DIR, SMOKE_OUTPUT_DIR, SMOKE_SFT_DIR, SMOKE_GRPO_DIR,
    GRPO_FROM_BASE_DIR, GRPO_FROM_SFT_DIR, SFT_OUTPUT_DIR, PLOTS_DIR, PLOTS_ABLATION_DIR,
)
for d in _mkdirs:
    d.mkdir(parents=True, exist_ok=True)

if not TRAIN_SFT_SCRIPT.is_file():
    raise FileNotFoundError(f"train_sft.py not found at {TRAIN_SFT_SCRIPT}")
if not TRAIN_GRPO_SCRIPT.is_file():
    raise FileNotFoundError(f"train_grpo.py not found at {TRAIN_GRPO_SCRIPT} (cwd={Path.cwd()})")


def _train_grpo_cmd(*cli_args):
    """Invoke training/train_grpo.py with the same interpreter (subprocess helpers)."""
    import sys as _sys

    return [_sys.executable, str(TRAIN_GRPO_SCRIPT), *cli_args]


def _train_sft_cmd(*cli_args):
    import sys as _sys

    return [_sys.executable, str(TRAIN_SFT_SCRIPT), *cli_args]


print(f"REPO_DIR          : {REPO_DIR}")
print(f"TRAIN_GRPO_SCRIPT : {TRAIN_GRPO_SCRIPT}")
print(f"OUTPUT_DIR        : {OUTPUT_DIR}")
print(f"PLOTS_DIR         : {PLOTS_DIR}")
print(f"USE_GROUNDED_CURRICULUM : {USE_GROUNDED_CURRICULUM}")
print(f"STATIC_GROUNDED_DATASET : {STATIC_GROUNDED_DATASET}  (False => live materialized + 6-role packs + CurriculumManager)")
print(f"RELAX_ROSTER            : {RELAX_ROSTER}  (False => strict roster asserts in train_grpo)")
print(f"RUN_SFT_PHASE          : {RUN_SFT_PHASE}  → {SFT_OUTPUT_DIR}")
print(f"RUN_SMOKE_PIPELINE     : {RUN_SMOKE_PIPELINE}  (§8: 1 SFT + 1 GRPO)")
print(f"RUN_ABLATION_MAIN      : {RUN_ABLATION_MAIN}  (§9: full SFT + GRPO base + GRPO SFT)")
print(f"GRPO dirs              : {GRPO_FROM_BASE_DIR.name} | {GRPO_FROM_SFT_DIR.name}")
print(f"GRPO_ABLATION_MAX_STEPS: {GRPO_ABLATION_MAX_STEPS!r}  SFT_MAX_STEPS={SFT_MAX_STEPS}")
print(f"HIGH_VRAM_MODE         : {HIGH_VRAM_MODE}  BATCH_SIZE={BATCH_SIZE}  GPUs={_N_GPU}  parallel_grpo={ABLATION_PARALLEL_GRPO}")

## 1. Environment variables

Run **§0 Config** first so `STATIC_GROUNDED_DATASET` and `RELAX_ROSTER` exist; this cell syncs them into `os.environ` for the whole kernel.

In [ ]:
import os, socket, time

os.environ["MASTER_ADDR"]                = "127.0.0.1"
os.environ["MASTER_PORT"]                = str(30000 + (os.getpid() % 2000))
os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
os.environ["PIP_NO_CACHE_DIR"]           = "1"
os.environ["NCCL_DEBUG"]                 = "INFO"
os.environ["NCCL_IB_DISABLE"]            = "1"
os.environ["NCCL_P2P_DISABLE"]           = "0"
os.environ["OMP_NUM_THREADS"]            = "4"
os.environ["MKL_NUM_THREADS"]            = "4"
os.environ["PYTHONUNBUFFERED"]           = "1"
# CRITICAL: disable torch.compile to avoid Dynamo errors with GRPO
os.environ["TORCH_COMPILE_DISABLE"]      = "1"
# OpenEnv hackathon winners (kube-sre-gym): allocator + TRL rollout noise
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TRL_EXPERIMENTAL_SILENCE", "1")
# Unsloth HF probe (120s) before model load — not your model; error text is a generic template.
# train_grpo.py also sets this if unset. Set to "0" only if you want Unsloth telemetry + HF check.
os.environ.setdefault("UNSLOTH_DISABLE_STATISTICS", "1")
# Match §0 Config (run Config before this cell): live materialized vs static list; strict vs relaxed roster.
if STATIC_GROUNDED_DATASET:
    os.environ["ATC_STATIC_GROUNDED_DATASET"] = "1"
else:
    os.environ.pop("ATC_STATIC_GROUNDED_DATASET", None)
if RELAX_ROSTER:
    os.environ["ATC_RELAX_ROSTER"] = "1"
else:
    os.environ.pop("ATC_RELAX_ROSTER", None)
print(f"Synced ATC_* : STATIC_GROUNDED={STATIC_GROUNDED_DATASET}  RELAX_ROSTER={RELAX_ROSTER}")
os.environ["WORLD_SIZE"]                 = "1"
os.environ["RANK"]                       = "0"
os.environ["LOCAL_RANK"]                 = "0"
os.environ["LOCAL_WORLD_SIZE"]           = "1"
os.environ["PYTHONPATH"]                 = str(REPO_DIR) + ":" + os.environ.get("PYTHONPATH", "")

print(f"Hostname       : {socket.gethostname()}")
print(f"Start time     : {time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"MASTER_PORT    : {os.environ['MASTER_PORT']}")
print(f"TORCH_COMPILE_DISABLE=1  (Dynamo disabled)")

## 2. GPU check

In [ ]:
import subprocess, sys

subprocess.run(["nvidia-smi"], check=False)
print(f"Python: {sys.version}")
try:
    import torch
    n = torch.cuda.device_count() if torch.cuda.is_available() else 0
    print(f"CUDA devices: {n}  (§9 uses parallel GRPO when ABLATION_PARALLEL_GRPO and n>=2)")
except Exception as exc:
    print(f"CUDA probe skipped: {exc}")

## 3. Load W&B key from .env (if present)

In [ ]:
import re

def _load_dotenv(path):
    """Minimal .env loader — no external deps needed."""
    p = Path(path)
    if not p.exists():
        return
    for line in p.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        m = re.match(r'^(?:export\s+)?([A-Za-z_][A-Za-z0-9_]*)\s*=\s*(.*)$', line)
        if m:
            k, v = m.group(1), m.group(2).strip().strip('"\'')
            if k not in os.environ:   # don't overwrite already-set vars
                os.environ[k] = v

for env_file in (REPO_DIR / ".env", REPO_DIR / ".env.wandb"):
    _load_dotenv(env_file)
    if env_file.exists():
        print(f"Loaded: {env_file}")

# Explicit key in config cell takes priority
if WANDB_KEY.strip():
    os.environ["WANDB_API_KEY"] = WANDB_KEY.strip()

# Normalize key: strip assignment prefix + whitespace (common .env mistake)
raw_key = os.environ.get("WANDB_API_KEY", "")
raw_key = raw_key.lstrip("\r").split("=")[-1].strip()
if raw_key:
    os.environ["WANDB_API_KEY"]  = raw_key
    os.environ["WANDB_MODE"]     = "online"
    os.environ["WANDB_PROJECT"]  = WANDB_PROJECT
    print(f"W&B key found (len={len(raw_key)}) → mode=online")
else:
    os.environ.pop("WANDB_API_KEY", None)
    os.environ["WANDB_MODE"] = "offline"
    print("W&B offline (no key)")

## 4. Clear stale Unsloth pycache

In [ ]:
import shutil

removed = 0
for p in REPO_DIR.rglob("__pycache__"):
    try:
        shutil.rmtree(p)
        removed += 1
    except Exception:
        pass
print(f"Cleared {removed} __pycache__ dirs")

## 5. Install constraint-compatible packages

This cell follows package metadata constraints for the **locked preinstalled runtime**.
It avoids pins that conflict with `transformers==5.5.4` (notably `huggingface-hub==0.36.2`).

In [ ]:
import subprocess, sys
from importlib.metadata import version, PackageNotFoundError
from packaging.version import Version


def pip(*args):
    cmd = [sys.executable, "-m", "pip"] + list(args)
    print("+", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=False)
    if result.returncode != 0:
        raise RuntimeError(f"pip failed: {' '.join(args[:4])}")


def v(pkg, default="(not installed)"):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return default


print("Detected before install:")
for pkg in [
    "torch", "transformers", "accelerate", "peft", "bitsandbytes", "xformers",
    "trl", "huggingface-hub", "datasets", "multiprocess", "tokenizers",
]:
    print(f"  {pkg:<16} {v(pkg)}")

tf_ver = Version(v("transformers", "0"))

print("\nInstalling vllm for TRL GRPO import-time dependency...")
pip("install", "--upgrade", "--no-input", "vllm==0.19.1")

# Core fix for your traceback: transformers 5.5.4 requires huggingface-hub >= 1.5.0,<2.0
print("\nInstalling runtime-compatible utility deps...")
pip(
    "install", "--upgrade", "--no-input",
    "huggingface-hub>=1.5.0,<2.0",
    "hf_transfer==0.1.9",
    "datasets==4.8.4",
    "multiprocess==0.70.19",
    "xxhash==3.6.0",
    "tyro==0.9.17",
    "wandb==0.19.11",
)

# Keep TRL explicit for notebook behavior consistency.
pip("install", "--upgrade", "--no-input", "trl==0.16.0")

# Unsloth 2026.4.8 metadata caps transformers at <= 5.5.0.
# With the locked transformers==5.5.4 runtime, the published wheel is not metadata-compatible.
print(
    "\nUnsloth install is blocked by locked transformers="
    + str(tf_ver)
    + "; published unsloth 2026.4.8 metadata requires <= 5.5.0."
)

print("\nDone.")

## 6. Verify installations

In [ ]:
import sys
from importlib.metadata import version, PackageNotFoundError
from packaging.version import Version


def v(pkg, default="(not installed)"):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return default


# Force-reimport after pip install in same process
for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ("unsloth", "trl", "peft", "accelerate", "bitsandbytes", "transformers", "huggingface_hub", "vllm")):
        del sys.modules[mod]

import torch
import transformers
import huggingface_hub
import vllm
import trl

print(f"PyTorch        : {torch.__version__}")
print(f"Transformers   : {transformers.__version__}")
print(f"vLLM           : {vllm.__version__}")
print(f"TRL            : {trl.__version__}")
print(f"HF Hub         : {huggingface_hub.__version__}")
print(f"Datasets       : {v('datasets')}")
print(f"Multiprocess   : {v('multiprocess')}")
print(f"CUDA           : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

tf_ver = Version(transformers.__version__)
try:
    import unsloth
    from unsloth import FastLanguageModel
    print(f"Unsloth        : {unsloth.__version__}")
except Exception as exc:
    print(
        "Unsloth import failed under the locked runtime (transformers="
        f"{tf_ver}). Published unsloth 2026.4.8 metadata requires transformers<=5.5.0. "
        f"Error: {exc}"
    )

from trl import GRPOConfig, GRPOTrainer
print("All core imports OK")

## 7. W&B login

In [ ]:
if os.environ.get("WANDB_MODE") == "online":
    try:
        import wandb
        wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)
        print(f"W&B logged in  project={WANDB_PROJECT}")
    except Exception as exc:
        print(f"W&B login failed ({exc}) → switching to offline")
        os.environ["WANDB_MODE"] = "offline"
else:
    print("W&B offline")

## 8. Smoke — 1×SFT + 1×GRPO

When **`RUN_SMOKE_PIPELINE`** is True (§0): tiny **SFT** (`--max_steps 1`) into `SMOKE_SFT_DIR`, then **GRPO** (`--max_steps 1`, `--adapter_in` smoke SFT) into `SMOKE_GRPO_DIR`. Uses **`SMOKE_MODEL`** for speed. Disable with **`RUN_SMOKE_PIPELINE = False`**.

Grounded flags match §9; `ATC_*` env comes from §1.

In [ ]:
import subprocess

if not RUN_SMOKE_PIPELINE:
    print("§8 smoke skipped (RUN_SMOKE_PIPELINE=False)")
else:
    print(f"===== §8 SMOKE: 1 SFT + 1 GRPO (model={SMOKE_MODEL}) =====")
    sft_smoke_args = [
        "--model", SMOKE_MODEL,
        "--output_dir", str(SMOKE_SFT_DIR),
        "--n_episodes", "8",
        "--max_steps", "1",
        "--batch_size", "1",
        "--grad_accum", "1",
        "--seed", str(SEED),
    ]
    if CURRICULUM_STATE:
        sft_smoke_args.extend(["--curriculum_state", str(CURRICULUM_STATE)])
    r0 = subprocess.run(_train_sft_cmd(*sft_smoke_args), env=os.environ, cwd=str(REPO_DIR))
    if r0.returncode != 0:
        raise RuntimeError(f"Smoke SFT FAILED (exit {r0.returncode})")
    grpo_smoke = [
        "--model", SMOKE_MODEL,
        "--output_dir", str(SMOKE_GRPO_DIR),
        "--adapter_in", str(SMOKE_SFT_DIR),
        "--episodes", "1",
        "--n_generations", "2",
        "--batch_size", "6",
        "--grad_accum", "1",
        "--seed", str(SEED),
        "--no_eval",
        "--max_steps", "1",
    ]
    if USE_GROUNDED_CURRICULUM:
        grpo_smoke.append("--grounded_curriculum")
        if STATIC_GROUNDED_DATASET:
            grpo_smoke.append("--static_grounded_dataset")
        if CURRICULUM_STATE:
            grpo_smoke.extend(["--curriculum_state", str(CURRICULUM_STATE)])
    smoke_env = dict(os.environ)
    smoke_env["ATC_STRICT_GATES"] = "1" if STRICT_GATES else "0"
    if STATIC_GROUNDED_DATASET:
        smoke_env["ATC_STATIC_GROUNDED_DATASET"] = "1"
    else:
        smoke_env.pop("ATC_STATIC_GROUNDED_DATASET", None)
    if RELAX_ROSTER:
        smoke_env["ATC_RELAX_ROSTER"] = "1"
    else:
        smoke_env.pop("ATC_RELAX_ROSTER", None)
    r1 = subprocess.run(_train_grpo_cmd(*grpo_smoke), env=smoke_env, cwd=str(REPO_DIR))
    if r1.returncode != 0:
        raise RuntimeError(f"Smoke GRPO FAILED (exit {r1.returncode})")
    print("===== §8 SMOKE COMPLETE =====")

## 9. Ablation — full SFT, then GRPO from base vs from SFT

In [ ]:
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

print("===== §9 ABLATION =====")
print(f"Model  : {TRAIN_MODEL}  EPISODES={EPISODES}  N_GEN={N_GENERATIONS}  batch/accum={BATCH_SIZE}/{GRAD_ACCUM}")
print(f"SFT    : max_steps={SFT_MAX_STEPS} → {SFT_OUTPUT_DIR}")
print(f"GRPO   : cap={GRPO_ABLATION_MAX_STEPS!r}  base={GRPO_FROM_BASE_DIR.name}  sft_init={GRPO_FROM_SFT_DIR.name}")
print()

if not RUN_ABLATION_MAIN:
    print("RUN_ABLATION_MAIN=False — skipping §9")
else:
    train_env = dict(os.environ)
    train_env["ATC_STRICT_GATES"] = "1" if STRICT_GATES else "0"
    if STATIC_GROUNDED_DATASET:
        train_env["ATC_STATIC_GROUNDED_DATASET"] = "1"
    else:
        train_env.pop("ATC_STATIC_GROUNDED_DATASET", None)
    if RELAX_ROSTER:
        train_env["ATC_RELAX_ROSTER"] = "1"
    else:
        train_env.pop("ATC_RELAX_ROSTER", None)

    if RUN_SFT_PHASE:
        print("===== SFT (full JSON imitation) =====")
        sft_args = [
            "--model", TRAIN_MODEL,
            "--output_dir", str(SFT_OUTPUT_DIR),
            "--n_episodes", str(SFT_N_EPISODES),
            "--max_steps", str(SFT_MAX_STEPS),
            "--batch_size", str(SFT_BATCH),
            "--grad_accum", str(SFT_GRAD_ACCUM),
            "--seed", str(SEED),
        ]
        if CURRICULUM_STATE:
            sft_args.extend(["--curriculum_state", str(CURRICULUM_STATE)])
        sft_res = subprocess.run(_train_sft_cmd(*sft_args), env=os.environ, cwd=str(REPO_DIR))
        if sft_res.returncode != 0:
            raise RuntimeError(f"SFT FAILED (exit {sft_res.returncode})")
        print("===== SFT COMPLETE =====")

    def _grpo_cmd(out_dir, adapter_in=None):
        cmd = [
            "--model", TRAIN_MODEL,
            "--output_dir", str(out_dir),
            "--episodes", str(EPISODES),
            "--n_generations", str(N_GENERATIONS),
            "--batch_size", str(BATCH_SIZE),
            "--grad_accum", str(GRAD_ACCUM),
            "--max_new_tokens", str(MAX_NEW_TOKENS),
            "--temperature", str(TEMPERATURE),
            "--logging_steps", str(LOGGING_STEPS),
            "--eval_episodes", str(EVAL_EPISODES),
            "--seed", str(SEED),
        ]
        if adapter_in is not None:
            cmd.extend(["--adapter_in", str(adapter_in)])
        if GRPO_ABLATION_MAX_STEPS is not None:
            cmd.extend(["--max_steps", str(int(GRPO_ABLATION_MAX_STEPS))])
        if USE_GROUNDED_CURRICULUM:
            cmd.append("--grounded_curriculum")
            if STATIC_GROUNDED_DATASET:
                cmd.append("--static_grounded_dataset")
            if CURRICULUM_STATE:
                cmd.extend(["--curriculum_state", str(CURRICULUM_STATE)])
        return cmd

    cmd_base = _grpo_cmd(GRPO_FROM_BASE_DIR, adapter_in=None)
    _sft_adapt = SFT_OUTPUT_DIR if (SFT_OUTPUT_DIR / "adapter_config.json").is_file() else None
    if _sft_adapt is None and RUN_SFT_PHASE:
        raise RuntimeError("SFT finished but adapter_config.json missing under SFT_OUTPUT_DIR")
    cmd_sft = _grpo_cmd(GRPO_FROM_SFT_DIR, adapter_in=_sft_adapt)

    t0 = time.monotonic()
    if ABLATION_PARALLEL_GRPO:
        print("===== GRPO parallel (CUDA 0 = base, CUDA 1 = SFT init) =====")

        def _run(dev, cmd, tag):
            env = {**train_env, "CUDA_VISIBLE_DEVICES": str(dev)}
            r = subprocess.run(_train_grpo_cmd(*cmd), env=env, cwd=str(REPO_DIR))
            return tag, r.returncode

        with ThreadPoolExecutor(max_workers=2) as ex:
            futs = [
                ex.submit(_run, 0, cmd_base, "grpo-from-base"),
                ex.submit(_run, 1, cmd_sft, "grpo-from-sft"),
            ]
            for fu in as_completed(futs):
                tag, code = fu.result()
                if code != 0:
                    raise RuntimeError(f"GRPO {tag} FAILED (exit {code})")
    else:
        print("===== GRPO sequential: base LoRA first =====")
        r2 = subprocess.run(_train_grpo_cmd(*cmd_base), env=train_env, cwd=str(REPO_DIR))
        if r2.returncode != 0:
            raise RuntimeError(f"GRPO base FAILED (exit {r2.returncode})")
        print("===== GRPO sequential: SFT-init LoRA =====")
        r3 = subprocess.run(_train_grpo_cmd(*cmd_sft), env=train_env, cwd=str(REPO_DIR))
        if r3.returncode != 0:
            raise RuntimeError(f"GRPO SFT-init FAILED (exit {r3.returncode})")
    elapsed = time.monotonic() - t0
    print(f"===== §9 ABLATION COMPLETE ({elapsed/60:.1f} min) =====")

## 10. Generate plots (per run + ablation overlay)

In [ ]:
import json
import sys

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import matplotlib
matplotlib.use("Agg")
from training.plot_rewards import (
    plot_all_training_artifacts,
    plot_ablation_eval_bars,
    plot_ablation_composite_overlay,
)

PLOTS_ABLATION_DIR.mkdir(parents=True, exist_ok=True)

ablation_runs = [
    ("grpo_from_base", GRPO_FROM_BASE_DIR),
    ("grpo_from_sft", GRPO_FROM_SFT_DIR),
]
all_generated = []
for label, outd in ablation_runs:
    sub = PLOTS_DIR / label
    sub.mkdir(parents=True, exist_ok=True)
    if not outd.is_dir():
        print(f"[skip] {label}: missing {outd}")
        continue
    ps = plot_all_training_artifacts(outd, sub, show=False)
    all_generated.extend(ps)
    print(f"{label}: {len(ps)} plot(s) → {sub}")

# Ablation: shared baseline + both trained evals + composite overlay
base_path = GRPO_FROM_BASE_DIR / "base_model_metrics.json"
if not base_path.is_file():
    base_path = GRPO_FROM_SFT_DIR / "base_model_metrics.json"
trained_payloads = []
curve_payloads = []
for label, outd in ablation_runs:
    tm = outd / "trained_model_metrics.json"
    rc = outd / "reward_curves.json"
    if tm.is_file():
        trained_payloads.append((label.replace("_", " "), json.loads(tm.read_text())))
    if rc.is_file():
        curve_payloads.append((label.replace("_", " "), json.loads(rc.read_text())))
if base_path.is_file() and len(trained_payloads) >= 1:
    baseline = json.loads(base_path.read_text())
    pbar = plot_ablation_eval_bars(baseline, trained_payloads, PLOTS_ABLATION_DIR, show=False)
    if pbar:
        all_generated.append(pbar)
if len(curve_payloads) >= 2:
    pov = plot_ablation_composite_overlay(curve_payloads, PLOTS_ABLATION_DIR, show=False)
    if pov:
        all_generated.append(pov)

bundle = {"baseline_path": str(base_path), "runs": {}}
for label, outd in ablation_runs:
    bp, tp = outd / "base_model_metrics.json", outd / "trained_model_metrics.json"
    bundle["runs"][label] = {
        "dir": str(outd),
        "base": json.loads(bp.read_text()) if bp.is_file() else None,
        "trained": json.loads(tp.read_text()) if tp.is_file() else None,
    }
bundle_path = OUTPUT_DIR / "ablation_eval_bundle.json"
bundle_path.write_text(json.dumps(bundle, indent=2), encoding="utf-8")
print(f"Wrote {bundle_path}")

print(f"\nTotal {len(all_generated)} plot file(s); ablation → {PLOTS_ABLATION_DIR}")

## 11. Display plots

In [ ]:
from IPython.display import Image, display

for png in sorted(PLOTS_DIR.rglob("*.png")):
    rel = png.relative_to(PLOTS_DIR)
    print(f"── {rel} ──")
    display(Image(filename=str(png), width=900))

## 12. Output file summary

In [ ]:
for title, root in (
    ("SFT-init GRPO (default focus)", TRAIN_OUTPUT_DIR),
    ("GRPO from base", GRPO_FROM_BASE_DIR),
    ("Full SFT adapter", SFT_OUTPUT_DIR),
):
    print(f"=== {title}: {root} ===")
    if not root.is_dir():
        print("  (missing)")
        continue
    for f in sorted(root.rglob("*")):
        if f.is_file():
            size = f.stat().st_size
            unit = "KB" if size < 1_000_000 else "MB"
            val = size / 1_000 if size < 1_000_000 else size / 1_000_000
            print(f"  {f.relative_to(root):<48}  {val:6.1f} {unit}")
    print()

print(f"Plots (recursive): {PLOTS_DIR}")
for f in sorted(PLOTS_DIR.rglob("*.png")):
    print(f"  {f.relative_to(PLOTS_DIR)}")